# Exact Fidelity and Local Default Equivalence

## tl;dr

baseline: exact 46/56, default-equivalent 48/56; candidate: exact 48/56, default-equivalent 48/56.

Diagnostic only; defaults and the official Gate remain unchanged.

## Context & Methods

14 new local requests, two catalog orders, two trials, two unchanged adapters. Only the 1.5B/v1 setting is evaluated. Exact matching and default-equivalence were frozen before inference.

### Key Assumptions

The only allowed equivalence is optional priority omission versus normal in the pinned local ticket handler. No query translation, fuzzy matching, field removal or output-based matching. Source cases are not independently reviewed. Repeated orders/trials are not independent samples. Historical rescoring is a separate post-hoc addendum, not overwritten evidence.

## Data

### 1. Validate Sources and Recompute

Source: `artifacts/reference-workload/tool-default-semantics-diagnostic-v1.json`. The independent audit imports no application scoring code and does not run models.

In [1]:
import hashlib, json, sys
from pathlib import Path
root = Path.cwd()
while not (root / 'reference_workload/runtime_matrix.json').exists():
    assert root != root.parent
    root = root.parent
assert hashlib.sha256((root / 'artifacts/reference-workload/tool-default-semantics-diagnostic-v1.json').read_bytes()).hexdigest() == 'ed5c3b2304d89e6241f7e713480798b78f068565a690498271933d8c911abd50'
assert hashlib.sha256((root / 'tools/review_tool_default_semantics.py').read_bytes()).hexdigest() == '848f38fc511ff694b5cce7d06f54355764b03cc916709781e7368a1809aaf75a'
sys.path.insert(0, str(root / 'tools'))
from review_tool_default_semantics import audit
result = audit(root)
print({k: result[k] for k in ('assessment', 'observations', 'http_calls', 'fault_replay_traces')})

{'assessment': 'share_with_caveats_local_default_equivalence_only', 'observations': 112, 'http_calls': 168, 'fault_replay_traces': 336}


## Results

### 2. Fresh Requests, Accuracy and Costs

Token values are measured HTTP totals. Local median latency is descriptive, not an SLA.

In [2]:
print(json.dumps(result['summary'], indent=2))
print(json.dumps(result['within_candidate_transitions'], indent=2))

[
  {
    "variant": "baseline",
    "n": 56,
    "selection": 52,
    "exact": 46,
    "equivalent": 48,
    "default_only": 2,
    "prompt_tokens": 48600,
    "completion_tokens": 1768,
    "median_latency_ms": 833.2968499744311
  },
  {
    "variant": "candidate",
    "n": 56,
    "selection": 52,
    "exact": 48,
    "equivalent": 48,
    "default_only": 0,
    "prompt_tokens": 65352,
    "completion_tokens": 2499,
    "median_latency_ms": 1298.7198500195518
  }
]
{
  "exact:1->1": 42,
  "equivalent:1->1": 44,
  "exact:0->1": 6,
  "exact:1->0": 4,
  "equivalent:1->0": 4,
  "exact:0->0": 4,
  "equivalent:0->0": 4,
  "equivalent:0->1": 4
}


### 3. Explicit and Omitted Priorities

Keep default-only differences visible alongside strict mismatches.

In [3]:
print(json.dumps([r for r in result['strata'] if r['stratum'].startswith('ticket_')], indent=2))

[
  {
    "variant": "baseline",
    "stratum": "ticket_explicit_normal",
    "n": 4,
    "exact": 4,
    "equivalent": 4
  },
  {
    "variant": "candidate",
    "stratum": "ticket_explicit_normal",
    "n": 4,
    "exact": 4,
    "equivalent": 4
  },
  {
    "variant": "candidate",
    "stratum": "ticket_omitted_default",
    "n": 4,
    "exact": 4,
    "equivalent": 4
  },
  {
    "variant": "baseline",
    "stratum": "ticket_omitted_default",
    "n": 4,
    "exact": 2,
    "equivalent": 4
  },
  {
    "variant": "baseline",
    "stratum": "ticket_explicit_high",
    "n": 4,
    "exact": 4,
    "equivalent": 4
  },
  {
    "variant": "candidate",
    "stratum": "ticket_explicit_high",
    "n": 4,
    "exact": 0,
    "equivalent": 0
  },
  {
    "variant": "candidate",
    "stratum": "ticket_explicit_low",
    "n": 4,
    "exact": 4,
    "equivalent": 4
  },
  {
    "variant": "baseline",
    "stratum": "ticket_explicit_low",
    "n": 4,
    "exact": 4,
    "equivalent": 4
  }
]


### 4. Historical Addendum

No original metric is overwritten.

In [4]:
print(json.dumps(result['historical_addendum_summary'], indent=2))

[
  {
    "entry": "small-baseline",
    "cohort": "frozen",
    "variant": "candidate",
    "n": 24,
    "exact": 11,
    "equivalent": 11
  },
  {
    "entry": "small-baseline",
    "cohort": "challenge",
    "variant": "baseline",
    "n": 12,
    "exact": 6,
    "equivalent": 6
  },
  {
    "entry": "small-baseline",
    "cohort": "challenge",
    "variant": "candidate",
    "n": 12,
    "exact": 9,
    "equivalent": 9
  },
  {
    "entry": "medium-candidate",
    "cohort": "frozen",
    "variant": "candidate",
    "n": 24,
    "exact": 24,
    "equivalent": 24
  },
  {
    "entry": "medium-candidate",
    "cohort": "challenge",
    "variant": "baseline",
    "n": 12,
    "exact": 7,
    "equivalent": 7
  },
  {
    "entry": "medium-candidate",
    "cohort": "challenge",
    "variant": "candidate",
    "n": 12,
    "exact": 9,
    "equivalent": 11
  },
  {
    "entry": "prompt-variant",
    "cohort": "frozen",
    "variant": "candidate",
    "n": 24,
    "exact": 20,
    "equivalen

## Takeaways

Default equivalence avoids conflating an omitted default with a different execution value; it does not certify general semantic correctness. Real content, authorization and independent review remain future work. Do not promote the candidate automatically or hide exact-fidelity regressions.

In [5]:
target = root / 'docs/reports/2026-09-09_tool_default_semantics' / 'audit.json'
target.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Audit saved')

Audit saved
